# ANPR cascade: Colab GPU test

Runs the Indian ANPR pipeline on a Colab GPU (Runtime > Change runtime type > T4 GPU).

**Pipeline under test:** COCO vehicle detector -> ByteTrack tracker -> plate detector on each vehicle crop -> plate-to-vehicle association -> quality gate -> RepSVTR OCR -> per-track consensus with Indian plate grammar repair. Design and references are in `docs/ARCHITECTURE.md`.

**Order of the notebook**
1. Setup: clone the repo and install it.
2. Unit tests (CPU, seconds).
3. Fetch the published weights (release `weights-v1`, checked against SHA-256).
4. OCR check with the fine-tuned RepSVTR.
5. Test video download.
6. Vehicle-only live-stream benchmark.
7. What the tracker saw.
8. Detector input size: 640 vs 1280.
9. Full pipeline (all stages) on the clip.
10. Annotated video of the first 15 seconds.
10b. **Live stream, end to end**: the whole clip played like a camera, with live boxes and plate text in the notebook, then a replay.
11. Scoring against ground-truth plates, if you supply a list.

All sections run: the weights are published (see `docs/WEIGHTS.md`). The full pipeline and the live stream need a GPU; on a T4 the full cascade processes about a third of a 30 fps stream (see `docs/ARCHITECTURE.md`).

## 1. Setup

Checks that a GPU is attached, clones the repository and installs it in editable mode together with the runtime dependencies (`ultralytics` for the vehicle detector, `rfdetr` for the plate detector, `pyyaml`, `opencv-python-headless`). `BRANCH` selects which branch to test. The last lines add `src/` to the kernel's import path, because a notebook kernel that is already running does not see a package installed after it started.

In [ ]:
import os, subprocess, sys
BRANCH = "main"
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)
if not os.path.isdir("/content/ANPR"):
    !git clone -q --depth 1 -b {BRANCH} https://github.com/sushovanjena/ANPR.git /content/ANPR
%cd /content/ANPR
!pip -q install -e . ultralytics rfdetr pytest ruff
import importlib
sys.path.insert(0, "/content/ANPR/src")   # the running kernel does not see a fresh editable install
importlib.invalidate_caches()


## 2. Unit tests

Runs the test suite and the linter. Every stage is tested with fake models, so this needs no GPU and no weights and takes a few seconds. A clean pass here means the orchestration logic (tracking, association, consensus, grammar, streaming, config, weights fetching) works before any model is involved.

In [ ]:
!python -m pytest -q && ruff check .

## 3. Fetch published weights

Downloads the plate detector, OCR weights and dictionary listed in `weights/manifest.json` into `weights/`, verifying each SHA-256, and clones OpenOCR (the RepSVTR code base) into `third_party/`. Files that have not been published yet are skipped with a message, so this cell is safe to run before the weights exist. If you trained the weights yourself, copy them into `weights/` with the file names from the manifest instead.

In [ ]:
!python -m anpr.fetch_weights --openocr
!ls -la weights

## 4. OCR check with the fine-tuned RepSVTR

Builds the RepSVTR recogniser from the downloaded weights, which also confirms that the checkpoint loads strictly into the 37-class head, and reads one crop. By default it reads a random-noise crop, which only proves the model builds and decodes. Set `CROP` to a real plate image path to see a real read. Skipped automatically if the weights are not present.

In [ ]:
import cv2, numpy as np
from anpr import weights
from anpr.ocr.recognizer import PlateReader, RepSvtrRecognizer
try:
    ocr = RepSvtrRecognizer(str(weights.ensure_openocr("third_party/OpenOCR")),
                            str(weights.resolve("weights/ocr_repsvtr_v2.pt")),
                            str(weights.resolve("weights/plate_dict.txt")))
    CROP = None  # e.g. "/content/plate.jpg"
    img = cv2.imread(CROP) if CROP else np.random.randint(0, 255, (48, 192, 3), dtype=np.uint8)
    print(PlateReader(ocr)(img))
except weights.WeightsUnavailable as e:
    print("skipped:", e)

## 5. Test video

Downloads a 39-second, 1080p, 30 fps clip of dense traffic near Charminar, Hyderabad, from the Educational Technology Clearinghouse (ETC) at the University of South Florida. Videographer: Alexander Bear. Copyright Florida Center for Instructional Technology; see the [terms](https://etc.usf.edu/clipvideo/license) before reuse. Replace `VIDEO` with your own file or an RTSP URL to test other footage.

In [ ]:
VIDEO = "/content/charminar.m4v"
if not os.path.exists(VIDEO):
    !curl -sL -o {VIDEO} "https://etc.usf.edu/clipvideo/f/item/3272/dense-traffic-near-charminar-in-hyderabad.39.1920x1080.m4v"
import cv2
c = cv2.VideoCapture(VIDEO); print("fps", c.get(5), "frames", c.get(7), "size", c.get(3), c.get(4))

## 6. Vehicle-only live-stream benchmark

Runs the vehicle detector and tracker as a live stream. `--no-plates` skips the plate detector and OCR, so this works without plate weights. `realtime: true` in the config paces the file at its source FPS, so the run behaves like a camera: a slow pipeline skips frames instead of falling behind, and every skipped frame is counted.

How to read the metrics printed at the end:
- `frames_dropped` near zero means the pipeline keeps up with real time on this GPU.
- `vehicle_ms` and `frame_ms` give p50 and p95 latency per stage.
- Tracks are written as events to the JSONL file (one `final` event per vehicle, with its class and first and last seen times).

Models are warmed up before the stream opens. Without that, loading the detector on the first frame took about 16 s on a T4 and dropped about 480 frames.

In [ ]:
import json
!mkdir -p /content/out
out = subprocess.run("python -m anpr.run --config configs/default.yaml --source " + VIDEO + " --no-plates --camera-id charminar --jsonl /content/out/vehicles.jsonl", shell=True, capture_output=True, text=True).stdout
m = json.loads(out[out.rindex('{\n  "frames_in"'):]); print(json.dumps(m, indent=1))

## 7. What did the tracker see?

Summarises the event file: number of tracked vehicles per class. In dense Indian traffic the count here is a sanity check on recall. Expect many more two-wheelers and auto-rickshaws than a COCO detector reports: COCO has no auto-rickshaw class, so they are reported as `car` or `truck`.

In [ ]:
import collections
ev = [json.loads(l) for l in open("/content/out/vehicles.jsonl")]
print(len(ev), "tracks", dict(collections.Counter(e["vehicle_class"] for e in ev)))

## 8. Detector input size: 640 vs 1280

The default detector size is 640, which shrinks a 1080p frame about 3x and loses small vehicles. This cell repeats the benchmark at 640 and 1280 so you can see the trade-off between recall (tracks found) and speed (latency and drops).

Measured on a Colab T4 with this clip:

| imgsz | p50 / p95 frame ms | frames dropped | tracks | classes |
|---|---|---|---|---|
| 640 | 19 / 48 | 46 of 1172 | 28 | 15 truck, 13 car |
| 1280 | 23 / 63 | 96 of 1172 | 34 | 14 truck, 17 car, 3 motorcycle |
| 640 (second run, fresh runtime) | 16 / 37 | 5 of 1172 | 26 | 13 truck, 13 car |
| 1280 (second run) | 21 / 38 | 30 of 1172 | 34 | 17 car, 13 truck, 4 motorcycle |

Neither setting finds most of the two-wheelers visible in the clip, so a detector fine-tuned for Indian vehicles is the next improvement. Drops and latency vary from run to run (shared GPU, CPU decoding on a small VM); tracks and classes were stable. These are a few runs on one clip, not a benchmark.

In [ ]:
for size in (640, 1280):
    cfg = f"/content/cfg{size}.yaml"
    !sed 's/imgsz: 640/imgsz: {size}/' configs/default.yaml > {cfg}
    out = subprocess.run(f"python -m anpr.run --config {cfg} --source {VIDEO} --no-plates --jsonl /content/out/ev{size}.jsonl", shell=True, capture_output=True, text=True).stdout
    m = json.loads(out[out.rindex('{\n  "frames_in"'):])
    ev = [json.loads(l) for l in open(f"/content/out/ev{size}.jsonl")]
    print(size, {k: m[k] for k in ("frames_processed", "frames_dropped", "fps_processed")}, m["stages"]["frame_ms"])
    print("   tracks", len(ev), dict(collections.Counter(e["vehicle_class"] for e in ev)))

## 9. Full pipeline

Runs every stage: vehicle detection, tracking, plate detection, association, quality gate, OCR and consensus. Weights are fetched automatically (`auto_fetch_weights: true`); the published weights are downloaded on first use. Events are written to JSONL and SQLite. Each event is `confirmed` (a plate was read confidently, then the vehicle is no longer re-read) or `final` (the vehicle left the scene; the plate is `null` when none was read).

In [ ]:
from anpr import weights
try:
    for w in ("weights/plate_rfdetr_nano.pth", "weights/ocr_repsvtr_v2.pt", "weights/plate_dict.txt"):
        weights.resolve(w)
    ready = True
except weights.WeightsUnavailable as e:
    ready = False
    print("skipped: plate weights are not published yet.", e)
if ready:
    !python -m anpr.run --config configs/default.yaml --source {VIDEO} --jsonl /content/out/full.jsonl --sqlite /content/out/full.db


## 10. Annotated video: live detection and transcription

Runs the cascade over the first 15 seconds of the clip and plays an annotated copy inline. Each vehicle gets a coloured box with its track id and class; a yellow box marks the plate found on that vehicle; the black label under the vehicle is the running consensus plate text, which settles as more frames are read. The video is produced frame by frame (it is not paced to real time), so it is for looking at, not for timing.

`max_seconds` sets the length (None for the whole clip). Section 10b below runs the same cascade as a real-time stream.

In [ ]:
import base64
from IPython.display import HTML, display
from anpr import weights
from anpr.config import load_config
from anpr.factory import build_cascade
from anpr.viz import render_video, to_h264

try:
    cascade = build_cascade(load_config("configs/default.yaml"))
    print("mode: vehicles, plates and plate text")
except weights.WeightsUnavailable as e:
    print("plate weights not published yet:", e)
    print("mode: vehicles and tracks only")
    cascade = build_cascade(load_config("configs/default.yaml", {"plate.enabled": False}))

stats = render_video(VIDEO, "/content/out/annotated.mp4", cascade, max_seconds=15)
print(stats)
play = to_h264("/content/out/annotated.mp4")
b64 = base64.b64encode(play.read_bytes()).decode()
display(HTML(f'<video controls autoplay loop muted width="800" src="data:video/mp4;base64,{b64}"></video>'))


## 10b. Live stream, end to end

Runs the real streaming pipeline, not a batch renderer: the clip is opened like a camera and **paced at its own frame rate** (`realtime: true`), the grabber thread feeds a drop-oldest queue, and the inference loop runs vehicle detection, tracking, plate detection, OCR and consensus on each frame it can keep up with. A frame callback draws the tracked vehicles, plate boxes and the running plate text and updates the image below as the stream plays, with a status line under it (frame number, vehicles in view, plates read so far).

When the clip ends the runner flushes its events and prints the stage latencies and how many frames were dropped. The same frames are also recorded, repeating the last frame where the stream dropped some, so the replay lasts as long as the original clip. Plate events are written to `/content/out/live.jsonl`. Expect it to take about as long as the clip (39 seconds) plus model warm-up.

In [ ]:
import json
import warnings
import base64
import cv2
from IPython.display import HTML, Image, display, update_display
from anpr.config import load_config
from anpr.factory import build_runner
from anpr.viz import LiveViewer, to_h264

warnings.filterwarnings("ignore")  # hide library deprecation notices in the live output
cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
cap.release()

cfg = load_config("configs/default.yaml", {
    "source": VIDEO, "realtime": True, "camera_id": "charminar",
    "sinks.jsonl": "/content/out/live.jsonl",
})
display(HTML("starting: loading and warming the models..."), display_id="live-frame")
display(HTML(""), display_id="live-status")


def show(jpeg, status):
    update_display(Image(data=jpeg), display_id="live-frame")
    update_display(HTML(f"<pre style='font-size:14px'>{status}</pre>"), display_id="live-status")


viewer = LiveViewer(show, "/content/out/live.mp4", fps=fps)
runner = build_runner(cfg, on_frame=viewer)
viewer.cascade = runner.cascade
snapshot = runner.run()          # blocks until the clip ends, paced like a camera
viewer.close()
print(json.dumps(snapshot, indent=1))


### Replay of the live run

Plays the recording of the run above (the same annotated frames, in real time) and lists the plates the pipeline read.

In [ ]:
import base64
import json

from IPython.display import HTML, display
from anpr.viz import to_h264

play = to_h264("/content/out/live.mp4")
b64 = base64.b64encode(play.read_bytes()).decode()
display(HTML(f'<video controls autoplay loop muted width="800" src="data:video/mp4;base64,{b64}"></video>'))
events = [json.loads(l) for l in open("/content/out/live.jsonl")]
print(len(events), "plate events; plates read:", sorted({e["plate"] for e in events if e.get("plate")}))


## 11. Score against ground-truth plates

Compares the plates read from the video with a text file listing the plates you can read by eye (one per line). Reports recall, precision, character error rate (CER), vehicles with no read, and the share of reads that need human review. Set `GT_CSV` to your ground-truth file. Plates that a human cannot read from the video should not be listed.

In [ ]:
from anpr.eval.pipeline_eval import evaluate_events, load_events
GT_CSV = "/content/ground_truth_plates.txt"   # one plate per line
if os.path.exists(GT_CSV) and os.path.exists("/content/out/full.jsonl"):
    gt = [l.strip().upper() for l in open(GT_CSV) if l.strip()]
    print(evaluate_events(load_events("/content/out/full.jsonl"), gt))
else:
    print("skipped: needs", GT_CSV, "and the full-pipeline output")